# 낙상 후 확인 대화: Qwen3.5-2B LoRA SFT 재현
원본 실험은 연구실 GPU에서 수행했습니다. 이 노트북은 Colab용 이식본이며 당시 실행 로그가 아닙니다. 기본 실행은 GPU나 인증 없이 공개 데이터·분할 및 **이미 측정된** 결과를 확인합니다. 추가 학습은 기본적으로 꺼져 있습니다.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import json, hashlib, math, random, tempfile
RUN_TRAINING = False
DATA = 'https://huggingface.co/datasets/yeopeter1031/aiot-fall-dialogue-teacher-10k-r1/resolve/9082fca9a954148bba5deeee6cdb459b457414b2/'
MODEL = 'https://huggingface.co/yeopeter1031/aiot-fall-qwen3.5-2b-lora-r1/resolve/acd4d0a750731a600386d27c3969deb789dc6f1e/'
def fetch(root, path):
    with urlopen(root + path, timeout=60) as response:
        return response.read()
def dump(x): return json.dumps(x, ensure_ascii=False, separators=(',', ':'))

In [ ]:
split = json.loads(fetch(DATA, 'metadata/original-split.json'))
rows = {}
checks = {'train': 'd935ba19e2dffb7b5058f12769b7c803672b46154f3357a10f21c7b839d152c3','validation': 'a480d7cdf62893876fda8167d573b0ad20aa3256c35ddca084807899d25dd3ab'}
for name, sha in checks.items():
    b = fetch(DATA, f'sft/{name}.jsonl')
    assert hashlib.sha256(b).hexdigest() == sha
    rows[name] = list(map(json.loads, b.splitlines()))
    assert all(split[r['id']] == ('train' if name == 'train' else 'val') for r in rows[name])
assert [len(rows[n]) for n in checks] == [9464, 536]
assert len({r['id'] for part in rows.values() for r in part}) == 10000
first = lambda n: {dump(x['history'][0]) for x in rows[n]}
assert first('train').isdisjoint(first('validation'))
protocol = json.loads(fetch(MODEL, 'protocol.json'))
result = json.loads(fetch(MODEL, 'evaluation.json'))
for arm in ('original', 'sft'):
    s = result[arm]; print(f"{arm}: 공개 실험 행동 일치 {s['action']}/{s['n']}")
print('검증 완료: 학습 9464, 검증 536, 첫 턴 중복 없음')

## 선택: 새 어댑터 학습
원할 때만 GPU 런타임에서 `RUN_TRAINING=True`로 바꾸세요. BF16 지원 및 여유 VRAM 28 GiB 초과가 필요합니다(기존 실험의 시작 조건). 기본 무료 GPU에서 실행을 보장하지 않습니다. 의존성 설치 후 런타임 재시작을 요청받으면 같은 설정으로 다시 실행하세요. 기존 모델은 덮어쓰지 않습니다. 런타임 종료 전 출력 폴더를 별도로 보관하세요.

In [ ]:
if RUN_TRAINING:
    import sys, subprocess, torch
    assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), 'BF16 CUDA GPU가 필요합니다.'
    assert torch.cuda.mem_get_info()[0] > 28 * 2**30, '원 실험 사양에 맞는 GPU를 선택하세요.'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.24.0', 'peft==0.22.0', 'accelerate==1.16.0'])

In [ ]:
def run_sft(rows, protocol):
    import torch
    from transformers import AutoTokenizer, Qwen3_5ForConditionalGeneration, get_cosine_schedule_with_warmup
    from peft import LoraConfig, get_peft_model
    seed = 42
    random.seed(seed); torch.manual_seed(seed)
    base, rev = protocol['base'], protocol['base_revision']
    tok = AutoTokenizer.from_pretrained(base, revision=rev)
    def encode(r):
        messages = [{'role':'system', 'content':protocol['prompt']}, {'role':'user','content':dump({'history':r['history']})}]
        prefix = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        answer = dump(r['final']) + tok.eos_token
        prompt = tok.encode(prefix, add_special_tokens=False)
        ids = tok.encode(prefix + answer, add_special_tokens=False)
        assert ids[:len(prompt)] == prompt and tok.decode(ids[len(prompt):], skip_special_tokens=False) == answer
        assert ids[-1] == tok.eos_token_id and len(ids) < 1024
        return ids, [-100] * len(prompt) + ids[len(prompt):]
    train = list(map(encode, rows['train'])); val = list(map(encode, rows['validation']))
    model = Qwen3_5ForConditionalGeneration.from_pretrained(base, revision=rev, dtype=torch.bfloat16, device_map={'':'cuda:0'})
    model.config.use_cache = False
    model=get_peft_model(model,LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias='none',target_modules=r'.*\.language_model\.layers\.\d+\.(linear_attn\.(in_proj_qkv|in_proj_z|in_proj_a|in_proj_b|out_proj)|self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))'))
    params = [p for p in model.parameters() if p.requires_grad]
    assert sum(p.numel() for p in params) == 16819200
    random.shuffle(train)
    steps = math.ceil(len(train)/16)
    opt = torch.optim.AdamW(params, lr=1e-4, weight_decay=.01)
    sch = get_cosine_schedule_with_warmup(opt, 10, steps)
    D = Path(tempfile.mkdtemp(prefix='fall-sft-', dir=Path.cwd())); print('출력 폴더:',D)
    def loss(batch):
     n=max(len(ids) for ids,_ in batch);pad=tok.pad_token_id; X=[];Y=[];M=[]
     for ids,ys in batch:
      k=n-len(ids);X.append(ids+[pad]*k);Y.append(ys+[-100]*k);M.append([1]*len(ids)+[0]*k)
     x,y,m=[torch.tensor(a,device='cuda') for a in (X,Y,M)];return model(input_ids=x,labels=y,attention_mask=m,use_cache=False).loss
    for step,start in enumerate(range(0,len(train),16),1):
     b=train[start:start+16]; chunks=[b[i:i+4] for i in range(0,len(b),4)]; avg=0.;model.train()
     for chunk in chunks:
      v=loss(chunk)/len(chunks);v.backward();avg+=v.detach().item()
     norm=torch.nn.utils.clip_grad_norm_(params,1.,error_if_nonfinite=True).item();opt.step();sch.step();opt.zero_grad(set_to_none=True);met={'train/loss':avg,'train/grad_norm':norm,'train/step':step};assert math.isfinite(avg)
     if step%100==0 or step==steps:
      model.eval()
      with torch.no_grad():met['val/loss']=sum(loss(val[i:i+4]).item()*len(val[i:i+4]) for i in range(0,len(val),4))/len(val)
      model.save_pretrained(D/f'adapter-{step}');tok.save_pretrained(D/f'adapter-{step}')
     print(met) if step == 1 or step % 50 == 0 or step == steps else None
    return D

if RUN_TRAINING:
    output = run_sft(rows, protocol)
else:
    print('추가 학습을 실행하지 않았습니다.')

## 결과 읽는 법
검증 536건은 합성 전사에 대한 교사와의 일치도입니다. 상태 재확인만 예측해도 444/536입니다. R1은 경보 30/32, 질문 반복 0/19였습니다. 노트북의 기본 모드는 저장된 집계를 읽을 뿐, 추론을 새로 돌리거나 실기기 기능을 실행하지 않습니다.